# Full Diploma Experiment Pipeline

Run these cells from top to bottom in a GPU-enabled Google Colab runtime.

In [ ]:
!nvidia-smi

In [ ]:
!git clone https://github.com/aldaiq-aitu/facenet.git
%cd facenet
!pip install -r requirements.txt

## Provide your dataset

Place a legally obtained folder-per-identity dataset at `/content/raw_dataset`, or build the recommended CelebA-Light subset in the next optional cell.

In [ ]:
RAW_DATASET_DIR = '/content/raw_dataset'
ALIGNED_DIR = 'data/aligned'
SPLITS_DIR = 'data/splits'
VAL_PAIRS = 'data/pairs/val_pairs.csv'
TEST_PAIRS = 'data/pairs/test_pairs.csv'

## Optional: build the recommended CelebA-Light subset

If you have `img_align_celeba/` and `identity_CelebA.txt`, uncomment and run the next cell, then set `RAW_DATASET_DIR = 'data/celeba_light'`.

In [ ]:
# !python -m scripts.prepare_celeba_subset \
#   --images-dir /content/img_align_celeba \
#   --identity-file /content/identity_CelebA.txt \
#   --output-dir data/celeba_light \
#   --num-identities 1000 \
#   --images-per-identity 20 \
#   --min-images-per-identity 20
# RAW_DATASET_DIR = 'data/celeba_light'

In [ ]:
!python -m scripts.align_dataset --input-dir "$RAW_DATASET_DIR" --output-dir "$ALIGNED_DIR" --image-size 112
!python -m scripts.create_splits --input-dir "$ALIGNED_DIR" --output-dir "$SPLITS_DIR"
!python -m scripts.create_pairs --split-dir "$SPLITS_DIR/val" --output-csv "$VAL_PAIRS"
!python -m scripts.create_pairs --split-dir "$SPLITS_DIR/test" --output-csv "$TEST_PAIRS"

In [ ]:
!python -m scripts.run_all_experiments

In [ ]:
!python -m evaluation.compare_models \
  --pairs-csv "$TEST_PAIRS" \
  --checkpoint experiments/facenet/best.pt \
  --checkpoint experiments/mobilefacenet/best.pt \
  --checkpoint experiments/efficientnet_lite0/best.pt \
  --output-csv results/model_comparison.csv

In [ ]:
!python benchmark_checkpoints.py \
  --checkpoint experiments/facenet/best.pt \
  --checkpoint experiments/mobilefacenet/best.pt \
  --checkpoint experiments/efficientnet_lite0/best.pt \
  --output-csv results/benchmark_checkpoints.csv

!python plot_results.py \
  --metrics-csv results/model_comparison.csv \
  --benchmark-csv results/benchmark_checkpoints.csv

## Primary artifacts to save

- `experiments/**/best.pt`
- `experiments/**/history.csv`
- `results/model_comparison.csv`
- `results/benchmark_checkpoints.csv`
- `results/plots/*.png`